# **1. Import Library**
Pada tahap ini dilakukan import seluruh pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import joblib

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [ ]:
# Gunakan dataset hasil clustering yang memiliki fitur Target

df = pd.read_csv("data_clustering_inverse.csv")

In [32]:
# Tampilkan 5 baris pertama dengan function head

df.head()

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,CustomerAge_Bin,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,High,1
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,High,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,Low,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,Low,1
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,Low,1


## **Feature Encoding: One Hot Encoding**

In [ ]:
categorical_cols = list(df.select_dtypes(include=['object']).columns)

# Gunakan 'pd.get_dummies' untuk melakukan OneHotEncoding
df_encoded = pd.get_dummies(
    df,
    columns = categorical_cols,
    drop_first = True
)

# Tampilkan 5 baris pertama untuk memverifikasi hasilnya
df_encoded.head()

,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,Target,TransactionType_Debit,Location_Atlanta,Location_Austin,Location_Baltimore,...,Location_Tucson,Location_Virginia Beach,Location_Washington,Channel_Branch,Channel_Online,CustomerOccupation_Engineer,CustomerOccupation_Retired,CustomerOccupation_Student,CustomerAge_Bin_Low,CustomerAge_Bin_Medium
0,14.09,70.0,81.0,1.0,5112.21,1,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,376.24,68.0,141.0,1.0,13758.91,0,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,126.29,19.0,56.0,1.0,1122.35,1,True,False,False,False,...,False,False,False,False,True,False,False,True,True,False
3,184.50,26.0,25.0,1.0,8569.06,1,True,False,False,False,...,False,False,False,False,True,False,False,True,True,False
4,92.15,18.0,172.0,1.0,781.68,1,True,False,False,False,...,False,False,False,False,False,False,False,True,True,False


# **3. Data Splitting**
Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [34]:
# Menggunakan train_test_split() untuk melakukan pembagian dataset.

# Buat 'X' dengan menghapus 'Target' dari 'df_encoded' dan gunakan 'axis=1' untuk menandakan drop kolom.
X = df_encoded.drop('Target', axis=1)

# Buat 'y' dengan HANYA memilih kolom 'Target'.
y = df_encoded['Target']

# Panggil fungsi untuk membagi data.
#  - Gunakan 'stratify=y' agar proporsi kelas di train/test set sama.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Jumlah data total: ",len(X))
print("Jumlah data latih: ",len(X_train))
print("Jumlah data test: ",len(X_test))

Jumlah data total:  1945
Jumlah data latih:  1556
Jumlah data test:  389


# **4. Membangun Model Klasifikasi**
Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

In [ ]:
# Model klasifikasi menggunakan Decision Tree

# 1. Buat (instantiate) objek model Decision Tree
#    Gunakan 'random_state=42' agar hasilnya konsisten
decision_tree_model = DecisionTreeClassifier(random_state=42)

# 2. Latih (fit) model dengan data training (X_train dan y_train)
decision_tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [36]:
# Menyimpan Model

joblib.dump(decision_tree_model, 'decision_tree_model.h5')

['decision_tree_model.h5']

# **5. Membangun Model Klasifikasi**

In [ ]:
# Melatih model menggunakan algoritma klasifikasi scikit-learn 

# Buat (instantiate) objek model baru
rf_model = RandomForestClassifier(random_state=42)
lr_model = LogisticRegression(random_state=42)

# Latih (fit) model dengan data training (X_train dan y_train)
rf_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(random_state=42)

In [38]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.

# Buat prediksi pada data 'X_test' menggunakan kedua model
y_pred_dt = decision_tree_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_pred_lr = lr_model.predict(X_test)

# Tampilkan classification_report untuk Decision Tree
print("Decision Tree Performance")
print(classification_report(y_test, y_pred_dt))

print("="*50)

# Tampilkan classification_report untuk New Model
print("Random Forest Performance")
print(classification_report(y_test, y_pred_rf))

print("="*50)

print("Logistic Regression Performance")
print(classification_report(y_test, y_pred_lr))

Decision Tree Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

Random Forest Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

Logistic Regression Performance
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       196
           1       0.98      1.00      0.99       193

    accuracy                           0.99       389
   macro avg       0.99      0.99      0.99 

In [ ]:
# Menyimpan Model Selain Decision Tree

joblib.dump(rf_model, 'explore_random_forest_classification.h5')
joblib.dump(lr_model, 'explore_logistic_regression_classification.h5')

['explore_logistic_regression_classification.h5']

Hyperparameter Tuning Model

In [ ]:
# Lakukan Hyperparameter Tuning dan Latih ulang.
# Lakukan dalam satu cell ini saja.

# Tentukan Hyperparameter yang akan di-tuning
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
}

# Buat (instantiate) objek dari algoritma tuning
#  - 'estimator': Model yang akan di-tuning
#  - 'params': Hyperparameter yang sudah ditentukan
rf_model_tuned = GridSearchCV(
    estimator = RandomForestClassifier(random_state=42),
    param_grid = params,
    cv = 5,
    scoring = 'accuracy'
)

# Latih objek model dengan data training (X_train dan y_train)
rf_model_tuned.fit(X_train, y_train)

# Hyperparameter Logistic Regression
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

lr_model_tuned = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_params,
    cv=5,
    scoring='accuracy'
)

lr_model_tuned.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=42),
             param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'],
                         'solver': ['lbfgs']},
             scoring='accuracy')

In [41]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.

# Buat prediksi pada 'X_test' Gunakan model yang sudah di-tuning
y_pred_rf_tuning = rf_model_tuned.predict(X_test)
y_pred_lr_tuning = lr_model_tuned.predict(X_test)

# Tampilkan classification_report untuk model yang sudah di-tuning
print("Random Forest Tuned Model Performance")
print(classification_report(y_test, y_pred_rf_tuning))
print("="*50)
print("Logistic Regression Tuned Model Performance")
print(classification_report(y_test, y_pred_lr_tuning))

Random Forest Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

Logistic Regression Tuned Model Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389



In [42]:
# Menyimpan Model hasil tuning

joblib.dump(rf_model_tuned, 'random_forest_tuning_classification.h5')
joblib.dump(lr_model_tuned, 'logistic_regression_tuning_classification.h5')

['logistic_regression_tuning_classification.h5']